# mHC vs HC vs Baseline — Full T4 5000-iters Experiment

Notebook này chạy **Mức 3 full T4 5000 iterations** cho implementation mHC:

- Train from scratch 3 variants: **baseline**, **HC**, **mHC**
- Dataset: **FineWeb10B GPT-2 shards**
- GPU target: **NVIDIA Tesla T4 16GB**
- Precision: **float16**
- Model scale: full T4 config trong repo:
  - `block_size=1024`
  - `n_layer=6`
  - `n_head=6`
  - `n_embd=288`
  - `max_iters=5000`

Output chính sẽ nằm trong:

```text
reports/t4-full-5000iters/
```

Notebook này có cơ chế **skip variant đã xong** nếu đã có checkpoint + `summary.json`, giúp resume khi Colab/Kaggle bị rớt session.


## 0. Runtime checklist

Trước khi chạy:

1. Runtime phải có GPU T4.
2. Nên chạy trên Kaggle/Colab Pro nếu có thể, vì mHC full 5000 có thể rất lâu.
3. Nếu chạy Colab free, nên mount Google Drive để lưu repo/checkpoints/report tránh mất kết quả.
4. Không claim đây là full paper reproduction. Đây là **full T4 nanoGPT controlled comparison**, không phải BBH/MMLU/GSM8K benchmark.

Nếu notebook bị rớt giữa chừng, chạy lại từ đầu. Cell train sẽ tự skip variant đã hoàn thành nếu `FORCE_RETRAIN=False`.
5. **Rất quan trọng:** sửa `REPO_URL` và `REPO_BRANCH` ở cell config cho đúng fork/branch implementation của bạn. Notebook sẽ checkout branch/commit và in ra commit hash trước khi train.



In [ ]:
# =========================
# 1. Global configuration
# =========================

from pathlib import Path
import os
import subprocess
import json
import textwrap
import sys
import time

# ---------------------------------------------------------
# IMPORTANT: set your fork + branch here.
# ---------------------------------------------------------
# Use YOUR fork URL if your implementation changes are not on upstream.
# Example:
REPO_URL = "https://github.com/khoaoe/mHC-manifold-constrained-hyper-connections.git"

# Set the exact branch that contains your mHC implementation / T4 scripts.
REPO_BRANCH = "t4-inference-benchmark"

# Optional but recommended for reproducibility:
# This commit is the one recorded in your mini-run benchmark/training metadata.
# Keep it if you want to run the full experiment on the same code version.
# Set REPO_COMMIT = "" if you intentionally want the latest HEAD of REPO_BRANCH.
REPO_COMMIT = "a7dec54c156785513c7463b03bcbda7cae865645"

# Colab default. For Drive persistence, set this to something like:
REPO_DIR = Path("/content/drive/MyDrive/mhc")
# REPO_DIR = Path("/content/mhc")

# Main full experiment output.
REPORT_DIR = REPO_DIR / "reports" / "t4-full-5000iters"

# FineWeb10B shard count.
# 9 train shards + 1 validation shard is the repo default and is much better than 1 shard.
# If disk/time is too tight, you can reduce to 3, but report it clearly.
NUM_TRAIN_SHARDS = 9

# Full T4 training settings.
MAX_ITERS = 5000
EVAL_INTERVAL = 500
EVAL_ITERS = 50
BATCH_SIZE = 8
GRAD_ACCUM = 8
DEVICE = "cuda"
DTYPE = "float16"
WANDB_LOG = "False"
COMPILE_MODEL = "False"

# Benchmark settings.
BENCH_BATCH_SIZE = 1
PROMPT_LEN = 128
GEN_LEN = 32
NUM_WARMUP = 5
NUM_ITERS = 20
COMPILE_BENCH = "false"

# Safety switch:
# False = resume-friendly, skip variants that already have ckpt.pt + summary.json(ok=True)
# True  = delete old run dirs and retrain everything from scratch
FORCE_RETRAIN = False

print("REPO_URL:", REPO_URL)
print("REPO_BRANCH:", REPO_BRANCH)
print("REPO_COMMIT:", REPO_COMMIT or "<branch HEAD>")
print("REPO_DIR:", REPO_DIR)
print("REPORT_DIR:", REPORT_DIR)
print("FORCE_RETRAIN:", FORCE_RETRAIN)


## 1.1. Branch/commit policy

Notebook này **không còn clone ngầm default branch**. Bạn phải kiểm tra 3 dòng ở cell config:

```python
REPO_URL = "..."
REPO_BRANCH = "..."
REPO_COMMIT = "..."  # optional exact pin
```

Khuyến nghị: dùng **fork của bạn** + **branch implementation thật**. Nếu muốn reproduce đúng code của mini-run trước đó, giữ `REPO_COMMIT` theo commit hash đã ghi trong report. Nếu muốn chạy latest branch, đặt `REPO_COMMIT = ""`.


## 2. Optional: Mount Google Drive trên Colab

Chỉ chạy cell này nếu đang dùng Google Colab và muốn lưu checkpoint/report vào Drive.

Sau khi mount, sửa `REPO_DIR` ở cell config thành:

```python
REPO_DIR = Path("/content/drive/MyDrive/mhc")
```


In [ ]:
# Optional for Google Colab only.
# Uncomment if needed.

from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# =========================
# 3. Helper functions
# =========================

def run(cmd, cwd=None, env=None, check=True):
    """Run shell command with live output."""
    print("\n$ " + (cmd if isinstance(cmd, str) else " ".join(cmd)))
    return subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        shell=isinstance(cmd, str),
        check=check,
        text=True,
    )

def read_json(path: Path):
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text())
    except Exception:
        return None

def run_finished(run_dir: Path, min_iters: int = None) -> bool:
    """Return True if a training run looks completed enough for this experiment."""
    if min_iters is None:
        min_iters = MAX_ITERS
    ckpt = run_dir / "ckpt.pt"
    summary = read_json(run_dir / "summary.json")
    if not ckpt.exists() or not isinstance(summary, dict):
        return False
    if summary.get("ok") is not True:
        return False
    iter_num = int(summary.get("iter_num", 0) or 0)
    return iter_num >= min_iters

def print_run_status():
    nanogpt = REPO_DIR / "examples" / "nanogpt"
    for name, out_dir in [
        ("baseline", nanogpt / "out-t4-baseline"),
        ("hc", nanogpt / "out-t4-hc"),
        ("mhc", nanogpt / "out-t4-mhc"),
    ]:
        summary = read_json(out_dir / "summary.json")
        ckpt = out_dir / "ckpt.pt"
        status = "DONE" if run_finished(out_dir) else "NOT DONE"
        best = summary.get("best_val_loss") if isinstance(summary, dict) else None
        last_eval = summary.get("last_eval") if isinstance(summary, dict) else None
        print(f"{name:8s} | {status:8s} | ckpt={ckpt.exists()} | best_val_loss={best} | last_eval={last_eval}")

print("Helpers loaded.")


In [ ]:
# =========================
# 4. Clone / fetch / checkout the exact repo branch
# =========================

def git(cmd, check=True):
    return run(f"git -C {REPO_DIR} {cmd}", check=check)

if REPO_DIR.exists():
    print(f"Using existing path: {REPO_DIR}")
    if not (REPO_DIR / ".git").exists():
        raise RuntimeError(
            f"{REPO_DIR} exists but is not a git repo. "
            "Set REPO_DIR to a clean path or remove the directory."
        )
else:
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    run(f"git clone {REPO_URL} {REPO_DIR}")

# Make sure origin is the intended repo URL.
current_origin = subprocess.check_output(
    f"git -C {REPO_DIR} remote get-url origin",
    shell=True,
    text=True,
).strip()
print("Current origin:", current_origin)

if current_origin != REPO_URL:
    print("Updating origin URL to configured REPO_URL...")
    git(f"remote set-url origin {REPO_URL}")

# Fetch all branches/tags so branch checkout and optional commit pinning work.
git("fetch --all --tags --prune")

# Checkout the configured branch explicitly.
# If the branch already exists locally, switch to it; otherwise create a local tracking branch.
branch_exists = subprocess.run(
    f"git -C {REPO_DIR} rev-parse --verify {REPO_BRANCH}",
    shell=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
).returncode == 0

remote_branch_exists = subprocess.run(
    f"git -C {REPO_DIR} rev-parse --verify origin/{REPO_BRANCH}",
    shell=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
).returncode == 0

if branch_exists:
    git(f"checkout {REPO_BRANCH}")
elif remote_branch_exists:
    git(f"checkout -b {REPO_BRANCH} origin/{REPO_BRANCH}")
else:
    raise RuntimeError(
        f"Branch '{REPO_BRANCH}' not found in {REPO_URL}. "
        "Fix REPO_BRANCH in the config cell before training."
    )

# Pull latest branch HEAD unless we pin an exact commit below.
git(f"pull --ff-only origin {REPO_BRANCH}")

# Optional exact commit pin for reproducibility.
# This may detach HEAD, which is OK for an experiment.
if REPO_COMMIT:
    git(f"checkout {REPO_COMMIT}")

# Print verification info. Read this before starting 5000-iters training.
verified_branch = subprocess.check_output(
    f"git -C {REPO_DIR} rev-parse --abbrev-ref HEAD",
    shell=True,
    text=True,
).strip()
verified_commit = subprocess.check_output(
    f"git -C {REPO_DIR} rev-parse HEAD",
    shell=True,
    text=True,
).strip()
status_short = subprocess.check_output(
    f"git -C {REPO_DIR} status --short",
    shell=True,
    text=True,
).strip()

print("\n========== REPO VERIFICATION ==========")
print("origin :", subprocess.check_output(f"git -C {REPO_DIR} remote get-url origin", shell=True, text=True).strip())
print("branch :", verified_branch)
print("commit :", verified_commit)
print("dirty  :", bool(status_short))
if status_short:
    print(status_short)
print("=======================================\n")

if REPO_COMMIT and verified_commit != REPO_COMMIT:
    raise RuntimeError("Checked-out commit does not match REPO_COMMIT. Stop before training.")

# Basic sanity check for this implementation workflow.
required = [
    REPO_DIR / "examples" / "nanogpt" / "train.py",
    REPO_DIR / "examples" / "nanogpt" / "config" / "train_fineweb10B_t4.py",
    REPO_DIR / "examples" / "nanogpt" / "config" / "train_fineweb10B_hc_t4.py",
    REPO_DIR / "examples" / "nanogpt" / "config" / "train_fineweb10B_mhc_t4.py",
    REPO_DIR / "examples" / "nanogpt" / "run_t4_full_compare.sh",
    REPO_DIR / "examples" / "nanogpt" / "benchmark_inference.py",
    REPO_DIR / "examples" / "nanogpt" / "summarize_training_runs.py",
    REPO_DIR / "examples" / "nanogpt" / "summarize_benchmarks.py",
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required repo files:\n" + "\n".join(missing))

print("Repo sanity check passed.")


In [ ]:
# =========================
# 5. Install dependencies
# =========================

# Keep this explicit for Colab/Kaggle.
# -e installs repo package + local imports.
run(f"{sys.executable} -m pip install -q -U pip")
run(f"{sys.executable} -m pip install -q -e {REPO_DIR}")
run(f"{sys.executable} -m pip install -q huggingface_hub tiktoken einops pandas matplotlib")

print("Dependencies installed.")


In [ ]:
# =========================
# 6. GPU / runtime check
# =========================

import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Please switch runtime to GPU.")

gpu_name = torch.cuda.get_device_name(0)
total_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print("GPU:", gpu_name)
print(f"VRAM: {total_gb:.2f} GB")

if "T4" not in gpu_name:
    print("WARNING: This notebook is tuned for T4. It may still work, but report the actual GPU name.")


In [ ]:
# =========================
# 7. Download FineWeb10B GPT-2 shards
# =========================

nanogpt_dir = REPO_DIR / "examples" / "nanogpt"
data_dir = nanogpt_dir / "data" / "fineweb10B"

print("Data dir:", data_dir)
print(f"Downloading/checking {NUM_TRAIN_SHARDS} train shards + validation shard...")

run(
    f"{sys.executable} data/fineweb10B/download.py {NUM_TRAIN_SHARDS}",
    cwd=nanogpt_dir,
)

print("Available FineWeb files:")
for p in sorted(data_dir.glob("fineweb_*.bin"))[:20]:
    print(" -", p.name, f"{p.stat().st_size / (1024**2):.1f} MB")
print("Total .bin files:", len(list(data_dir.glob("fineweb_*.bin"))))


## 8. Train full T4 variants

Cell này chạy tuần tự:

1. baseline
2. HC
3. mHC

Mỗi variant dùng full T4 config 5000 iters.

Nếu `FORCE_RETRAIN=False`, notebook sẽ skip variant đã hoàn thành.  
Nếu muốn chạy lại từ đầu, set `FORCE_RETRAIN=True` ở cell config.


In [ ]:
# =========================
# 8. Train baseline / HC / mHC full 5000 iters
# =========================

import shutil

nanogpt_dir = REPO_DIR / "examples" / "nanogpt"
log_dir = nanogpt_dir / "logs"
log_dir.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

variants = [
    {
        "name": "baseline",
        "config": "config/train_fineweb10B_t4.py",
        "out_dir": "out-t4-baseline",
    },
    {
        "name": "hc",
        "config": "config/train_fineweb10B_hc_t4.py",
        "out_dir": "out-t4-hc",
    },
    {
        "name": "mhc",
        "config": "config/train_fineweb10B_mhc_t4.py",
        "out_dir": "out-t4-mhc",
    },
]

if FORCE_RETRAIN:
    for v in variants:
        path = nanogpt_dir / v["out_dir"]
        if path.exists():
            print("Deleting old run:", path)
            shutil.rmtree(path)

for v in variants:
    name = v["name"]
    out_dir = nanogpt_dir / v["out_dir"]
    log_path = log_dir / f"t4-full-5000-{name}.log"

    if not FORCE_RETRAIN and run_finished(out_dir):
        print(f"[SKIP] {name}: already completed at {out_dir}")
        continue

    print(f"\n[TRAIN] {name} -> {out_dir}")
    print(f"Log: {log_path}")

    # Use bash + tee for live logs and saved logs.
    cmd = f"""
    set -euo pipefail
    cd "{nanogpt_dir}"
    stdbuf -oL -eL {sys.executable} -u train.py "{v["config"]}" \
      "out_dir='{v["out_dir"]}'" \
      "max_iters={MAX_ITERS}" \
      "eval_interval={EVAL_INTERVAL}" \
      "eval_iters={EVAL_ITERS}" \
      "batch_size={BATCH_SIZE}" \
      "gradient_accumulation_steps={GRAD_ACCUM}" \
      "device='{DEVICE}'" \
      "dtype='{DTYPE}'" \
      "wandb_log={WANDB_LOG}" \
      "compile_model={COMPILE_MODEL}" \
      2>&1 | tee "{log_path}"
    """
    run(cmd)

    if not run_finished(out_dir):
        raise RuntimeError(f"{name} did not finish correctly. Check log: {log_path}")

print("\nTraining status:")
print_run_status()


## 9. Summarize training results

Tạo:

```text
reports/t4-full-5000iters/training_summary.csv
reports/t4-full-5000iters/training_summary.md
reports/t4-full-5000iters/training_summary.json
```


In [ ]:
# =========================
# 9. Summarize training runs
# =========================

run(
    f"""{sys.executable} examples/nanogpt/summarize_training_runs.py \
    --runs \
    baseline=examples/nanogpt/out-t4-baseline \
    hc=examples/nanogpt/out-t4-hc \
    mhc=examples/nanogpt/out-t4-mhc \
    --output-dir {REPORT_DIR}""",
    cwd=REPO_DIR,
)

print((REPORT_DIR / "training_summary.md").read_text()[:5000])


## 10. Benchmark inference

Benchmark này dùng synthetic token IDs, direct PyTorch, **không KV cache**.  
Dùng để so sánh overhead tương đối giữa baseline / HC / mHC trong cùng implementation, không phải production serving benchmark.


In [ ]:
# =========================
# 10. Benchmark inference
# =========================

bench_dir = REPORT_DIR / "benchmarks"
bench_dir.mkdir(parents=True, exist_ok=True)

ckpt_baseline = nanogpt_dir / "out-t4-baseline" / "ckpt.pt"
ckpt_hc = nanogpt_dir / "out-t4-hc" / "ckpt.pt"
ckpt_mhc = nanogpt_dir / "out-t4-mhc" / "ckpt.pt"

for ckpt in [ckpt_baseline, ckpt_hc, ckpt_mhc]:
    if not ckpt.exists():
        raise FileNotFoundError(f"Missing checkpoint: {ckpt}")

benchmark_variants = [
    ("baseline", ckpt_baseline, nanogpt_dir / "config" / "train_fineweb10B_t4.py"),
    ("hc", ckpt_hc, nanogpt_dir / "config" / "train_fineweb10B_hc_t4.py"),
    ("mhc", ckpt_mhc, nanogpt_dir / "config" / "train_fineweb10B_mhc_t4.py"),
]

for name, ckpt, config in benchmark_variants:
    print(f"\n[BENCH] {name}")
    run(
        f"""{sys.executable} {nanogpt_dir / "benchmark_inference.py"} \
        --ckpt {ckpt} \
        --config {config} \
        --device {DEVICE} \
        --dtype {DTYPE} \
        --batch-size {BENCH_BATCH_SIZE} \
        --prompt-len {PROMPT_LEN} \
        --gen-len {GEN_LEN} \
        --num-warmup {NUM_WARMUP} \
        --num-iters {NUM_ITERS} \
        --compile {COMPILE_BENCH} \
        --output-json {bench_dir / (name + ".json")} \
        --output-csv {bench_dir / (name + ".csv")} \
        --seed 1337""",
        cwd=REPO_DIR,
    )

run(f"{sys.executable} examples/nanogpt/summarize_benchmarks.py {bench_dir}", cwd=REPO_DIR)

print((bench_dir / "summary.md").read_text()[:5000])


## 11. Plot figures

Tạo figures trong:

```text
reports/t4-full-5000iters/figures/
```

Các hình nên dùng trong report:

- `val_loss_curve.png`
- `train_loss_curve.png`
- `best_val_loss_bar.png`
- `final_val_ppl_bar.png`
- `inference_tokens_per_sec_bar.png`
- `decode_ms_per_token_bar.png`


In [ ]:
# =========================
# 11. Generate report figures
# =========================

fig_dir = REPORT_DIR / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

run(
    f"""{sys.executable} examples/nanogpt/plot_training_comparison.py \
    --runs \
    baseline=examples/nanogpt/out-t4-baseline \
    hc=examples/nanogpt/out-t4-hc \
    mhc=examples/nanogpt/out-t4-mhc \
    --benchmark-summary {bench_dir / "summary.csv"} \
    --output-dir {fig_dir}""",
    cwd=REPO_DIR,
)

print("Figures:")
for p in sorted(fig_dir.glob("*.png")):
    print(" -", p)


## 12. Quick final diagnosis

Cell này đọc summary và in ra kết luận nhanh:

- baseline vs HC
- baseline vs mHC
- HC vs mHC
- overhead benchmark

Dùng đoạn này để kiểm tra trước khi đưa vào report.


In [ ]:
# =========================
# 12. Quick final diagnosis
# =========================

import pandas as pd
import math

train_csv = REPORT_DIR / "training_summary.csv"
bench_csv = REPORT_DIR / "benchmarks" / "summary.csv"

train_df = pd.read_csv(train_csv)
bench_df = pd.read_csv(bench_csv)

display(train_df[[
    "variant", "best_val_loss", "final_train_loss", "final_val_loss",
    "final_val_ppl", "tokens_seen", "iter_num", "elapsed_s"
]])

display(bench_df[[
    "variant", "model_parameter_count", "tokens_per_sec_mean",
    "full_context_generation_latency_per_token_ms_mean", "peak_vram_mb_max"
]])

def row(df, variant):
    return df[df["variant"] == variant].iloc[0]

b = row(train_df, "baseline")
h = row(train_df, "hc")
m = row(train_df, "mhc")

print("\n=== Training comparison ===")
print(f"Baseline final val loss: {b.final_val_loss:.4f}, PPL: {b.final_val_ppl:.2f}")
print(f"HC       final val loss: {h.final_val_loss:.4f}, PPL: {h.final_val_ppl:.2f}")
print(f"mHC      final val loss: {m.final_val_loss:.4f}, PPL: {m.final_val_ppl:.2f}")

print("\nImprovements over baseline:")
print(f"HC  final PPL improvement: {(b.final_val_ppl - h.final_val_ppl) / b.final_val_ppl * 100:.2f}%")
print(f"mHC final PPL improvement: {(b.final_val_ppl - m.final_val_ppl) / b.final_val_ppl * 100:.2f}%")

print("\nBest val loss:")
print(f"baseline: {b.best_val_loss:.4f}")
print(f"HC      : {h.best_val_loss:.4f}")
print(f"mHC     : {m.best_val_loss:.4f}")

bb = row(bench_df, "baseline")
bh = row(bench_df, "hc")
bm = row(bench_df, "mhc")

print("\n=== Inference benchmark ===")
print(f"Baseline tokens/sec: {bb.tokens_per_sec_mean:.2f}")
print(f"HC       tokens/sec: {bh.tokens_per_sec_mean:.2f}")
print(f"mHC      tokens/sec: {bm.tokens_per_sec_mean:.2f}")
print(f"HC slowdown vs baseline : {bb.tokens_per_sec_mean / bh.tokens_per_sec_mean:.2f}x")
print(f"mHC slowdown vs baseline: {bb.tokens_per_sec_mean / bm.tokens_per_sec_mean:.2f}x")
print(f"mHC slowdown vs HC      : {bh.tokens_per_sec_mean / bm.tokens_per_sec_mean:.2f}x")

print("\nRecommended report wording:")
print(textwrap.fill(
    "In the full T4 nanoGPT/FineWeb10B controlled experiment, we train baseline, HC, "
    "and mHC from scratch under the same model scale and training budget. We report "
    "validation loss/perplexity for model quality and synthetic full-context inference "
    "benchmark for runtime overhead. This remains a small-scale implementation study, "
    "not a full reproduction of the original paper's downstream benchmark suite.",
    width=100
))


## 13. Zip reports for download / submission

Tạo file:

```text
reports/t4-full-5000iters.zip
```

Nên nộp/attach các file chính:

- `training_summary.md`
- `benchmarks/summary.md`
- `figures/*.png`
- `training_summary.csv`
- `benchmarks/summary.csv`


In [ ]:
# =========================
# 13. Zip report directory
# =========================

zip_path = REPORT_DIR.with_suffix(".zip")
if zip_path.exists():
    zip_path.unlink()

run(f"zip -r {zip_path} {REPORT_DIR.name}", cwd=REPORT_DIR.parent)

print("Zipped report:", zip_path)
print("Size MB:", zip_path.stat().st_size / (1024**2))


## 14. Optional: copy report zip to Google Drive

Chỉ dùng nếu đang chạy Colab và đã mount Drive.


In [ ]:
# Optional: copy zip to Drive.
# Change destination if needed.

# drive_out = Path("/content/drive/MyDrive/mhc_reports")
# drive_out.mkdir(parents=True, exist_ok=True)
# shutil.copy2(zip_path, drive_out / zip_path.name)
# print("Copied to:", drive_out / zip_path.name)
